# Практическая работа 10. Задание 3
Составление расписания через раскраску графа конфликтов

In [1]:
# ребро - занятия нельзя проводить одновременно
graph = {
    'M1': ['M2', 'M3', 'P1'],
    'M2': ['M1', 'M3', 'P2'],
    'M3': ['M1', 'M2', 'P1', 'P2', 'P3'],
    'P1': ['M1', 'M3', 'P2', 'P3'],
    'P2': ['M2', 'M3', 'P1', 'P3'],
    'P3': ['M3', 'P1', 'P2']
}


def welsh_powell(graph):
    order = sorted(graph, key=lambda v: -len(graph[v]))
    colors = {}
    color = 0
    for v in order:
        if v in colors:
            continue
        colors[v] = color
        group = [v]
        for u in order:
            if u not in colors and all(u not in graph[x] for x in group):
                colors[u] = color
                group.append(u)
        color += 1
    return colors, order


def bron_kerbosch(R, P, X, graph, cliques):
    if not P and not X:
        cliques.append(R)
        return
    for v in list(P):
        bron_kerbosch(R | {v}, P & set(graph[v]), X & set(graph[v]),
                      graph, cliques)
        P.remove(v)
        X.add(v)

In [2]:
colors, order = welsh_powell(graph)
print('Порядок по степени:', [(v, len(graph[v])) for v in order])
k = max(colors.values()) + 1
print('Слотов (цветов):', k)
for c in range(k):
    print('  слот', c + 1, ':', [v for v in order if colors[v] == c])

# нижняя оценка: размер наибольшей клики
cliques = []
bron_kerbosch(set(), set(graph), set(), graph, cliques)
big = max(cliques, key=len)
print('Наибольшая клика:', sorted(big), ', значит слотов не меньше', len(big))

# проверка: у концов каждого ребра разные цвета
ok = True
for u in graph:
    for v in graph[u]:
        if colors[u] == colors[v]:
            ok = False
            print('Конфликт:', u, v)
print('Все ребра с разными цветами:', ok)
free = [v for v in graph if v not in colors]
print('Без слота осталось занятий:', len(free))

Порядок по степени: [('M3', 5), ('P1', 4), ('P2', 4), ('M1', 3), ('M2', 3), ('P3', 3)]
Слотов (цветов): 4
  слот 1 : ['M3']
  слот 2 : ['P1', 'M2']
  слот 3 : ['P2', 'M1']
  слот 4 : ['P3']
Наибольшая клика: ['M3', 'P1', 'P2', 'P3'] , значит слотов не меньше 4
Все ребра с разными цветами: True
Без слота осталось занятий: 0
